RNN for Text generation

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import urllib.request
import random

from torch.utils.data.dataset import _T_co
from torchvision import models

# Download tghe tiny Shakespeare Dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_name = "shakespeare.txt"

urllib.request.urlretrieve(url, file_name)

# Read the text
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Number of characters: {len(text)}")



Number of characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [8]:
# Get all unique characters
unique_chars = sorted(list(set(text)))

vocab_size = len(unique_chars)

print("Vocabulary size: ", vocab_size)
print("Unique Characters:")
print(unique_chars)

Vocabulary size:  65
Unique Characters:
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [9]:
# Create mappings between characters and Integers

# Character - > integer
char_to_idx = {char: i for i, char in enumerate(unique_chars)}

# Integer -> character
idx_to_char = {i: char for i, char in enumerate(unique_chars)}

# Convert the entire text to integers
encoded_text = [char_to_idx[char] for char in text]

# Preview the first 50 encoded texts
print(encoded_text[:50])



[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56]


In [10]:
# Create sequence for training
class TextDataset(Dataset):

    def __init__(self, encoded_text, seq_len = 100):
        self.data = encoded_text
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):

        x = self.data[index:index + self.seq_len]

        y = self.data[index + 1:index  + self.seq_len + 1]

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long),
        )

In [11]:
sequence_length = 100
batch_size = 64

dataset = TextDataset(encoded_text, sequence_length)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

print("Number of training sequences: ", len(dataset))


Number of training sequences:  1115294


In [13]:
# Define LSTM RNN
class CharLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim = 128, hidden_size = 256, num_layers = 2,):
        super(CharLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Character embeddings
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2,
        )

        # Output layer
        self.fc = nn.Linear(hidden_size, vocab_size,)

    def forward(self, x, hidden=None):

        # Convert character Ids to embeddings
        x = self.embedding(x)

        # Pass through LSTM
        output, hidden = self.lstm(x, hidden)

        # Predict next character
        output = self.fc(output)

        return output, hidden


In [14]:
# Create the Model
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Using device:", device)

model = CharLSTM(
    vocab_size = vocab_size,
    embedding_dim = 128,
    hidden_size = 256,
    num_layers = 2)

model = model.to(device)

print(model)



Using device: mps
CharLSTM(
  (embedding): Embedding(65, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=65, bias=True)
)


In [16]:
# Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)



In [18]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_num, (inputs, targets) in enumerate(dataloader):

        inputs = inputs.to(device)
        targets = targets.to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs, hidden = model(inputs)

        # Reshape outputs and targets
        outputs = outputs.reshape(-1, vocab_size)
        targets = targets.reshape(-1)

        # Calculate loss
        loss = criterion(outputs, targets)

        # Backpropagation
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5
        )

        optimizer.step()

        total_loss += loss.item()

        # Limit batches if training on CPU
        # Remove this block for full training
        if batch_num >= 500:
            break

    average_loss = total_loss / (batch_num + 1)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"- Loss: {average_loss:.4f}"
    )



In [ ]:
def generate_text(
    model,
    start_text,
    length=500,
    temperature=1.0
):

    model.eval()

    # Initial hidden state
    hidden = None

    generated_text = start_text

    # Feed starting characters into the model
    input_indices = [
        char_to_idx[c]
        for c in start_text
        if c in char_to_idx
    ]

    input_tensor = torch.tensor(
        input_indices,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        # Process starting text
        output, hidden = model(input_tensor, hidden)

        # Last character becomes next input
        current_input = input_tensor[:, -1:]

        for _ in range(length):

            output, hidden = model(
                current_input,
                hidden
            )

            # Get predictions for next character
            logits = output[:, -1, :]

            # Apply temperature scaling
            logits = logits / temperature

            # Convert logits to probabilities
            probabilities = F.softmax(
                logits,
                dim=-1
            )

            # Sample one character
            next_index = torch.multinomial(
                probabilities,
                num_samples=1
            )

            next_char = idx_to_char[
                next_index.item()
            ]

            generated_text += next_char

            # Use predicted character as next input
            current_input = next_index

    return generated_text

In [ ]:
generated = generate_text(
    model,
    start_text="ROMEO:",
    length=1000,
    temperature=0.8
)

print(generated)

In [ ]:
temperatures = [0.3, 0.7, 1.0, 1.5]

for temp in temperatures:

    print("\n" + "=" * 70)
    print(f"TEMPERATURE = {temp}")
    print("=" * 70)

    generated = generate_text(
        model,
        start_text="ROMEO:",
        length=300,
        temperature=temp
    )

    print(generated)